# Train Notebook

- Source: `src/train.py`
- 목적: 원본 파이썬 파일을 단계별로 실행/설명하기 위한 노트북 버전
- 실행 방법: 위에서 아래로 순서대로 실행


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # 노트북이 다른 경로에서 열렸을 때 프로젝트 루트 자동 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'src').exists() and (parent / 'configs').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


## Step 1. Setup and Imports

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
"""
학습 스크립트.

전체 흐름
─────────
1. configs/default.yaml (or defect.yaml) 로드
2. train / val DataLoader 생성
3. EfficientNet-B0 backbone + MLP head 모델 생성
4. [warm-up 단계] backbone 동결, head만 lr=1e-3으로 학습 (warmup_epochs)
5. [fine-tune 단계] backbone 동결 해제, 전체 함께 학습
6. 매 epoch마다 val macro-F1 측정, 개선되면 checkpoints/best_<task>.pth 저장
7. early_stop_patience epoch 동안 개선없으면 조기 종료

사용 예시
─────────
    # 로스팅 학습
    python -m src.train --config configs/default.yaml

    # 결점두 학습
    python -m src.train --config configs/defect.yaml

    # YAML 수정 없이 비삭를 CLI에서 당장 덮어는 방법
    python -m src.train --config configs/default.yaml --override model.name=resnet18 train.epochs=50
"""
from __future__ import annotations
import argparse
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.utils.logger import TBLogger
from src.dataset import SingleTaskDataset, get_transforms, compute_class_weights
from src.model import CoffeeClassifier
from src.losses import cross_entropy_loss


## Step 2. Function: parse_overrides

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def parse_overrides(items: list[str]) -> dict[str, Any]:
    out: dict[str, Any] = {}
    for it in items:
        k, v = it.split("=", 1)
        try:
            # CLI에서 들어온 문자열을 숫자/bool/list 등으로 최대한 복원한다.
            # 예: train.epochs=8, model.pretrained=False
            v = eval(v, {"__builtins__": {}}, {})
        except Exception:
            pass
        out[k] = v
    return out


## Step 3. Function: apply_overrides

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def apply_overrides(cfg: dict, overrides: dict) -> dict:
    # dotted key(train.epochs 같은 형태)를 실제 중첩 dict에 반영한다.
    for k, v in overrides.items():
        cur = cfg
        keys = k.split(".")
        for kk in keys[:-1]:
            cur = cur.setdefault(kk, {})
        cur[keys[-1]] = v
    return cfg


## Step 4. Function: make_loader

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def make_loader(csv: str, task: str, classes: list[str], img_size: int,
                train: bool, batch_size: int, num_workers: int):
    # config에 적힌 클래스 순서를 그대로 인덱스로 쓰면
    # train/eval/infer/streamlit이 모두 같은 라벨 체계를 공유할 수 있다.
    c2i = {c: i for i, c in enumerate(classes)}
    ds = SingleTaskDataset(
        csv_path=csv, task=task, class_to_idx=c2i,
        transform=get_transforms(task=task, train=train, img_size=img_size),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=train,
                      num_workers=num_workers, pin_memory=False), c2i


## Step 5. Function: evaluate

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def evaluate(model, loader, device) -> dict:
    # train.py 안에서는 검증셋에서 acc, macro F1만 빠르게 확인한다.
    # 자세한 리포트와 confusion matrix는 evaluate.py에서 별도로 만든다.
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            ps.append(logits.argmax(1).cpu())
            ys.append(y)
    y_true = torch.cat(ys).numpy()
    y_pred = torch.cat(ps).numpy()
    return {
        "acc": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


## Step 6. Function: main

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--override", nargs="*", default=[])
    args = ap.parse_args()

    # 1) 설정 파일 로드
    # 2) 필요하면 CLI override 반영
    # 3) seed 고정으로 재현성 확보
    cfg = load_config(args.config)
    cfg = apply_overrides(cfg, parse_overrides(args.override))
    set_seed(cfg["seed"])

    device = "cuda" if torch.cuda.is_available() else "cpu"
    # AMP는 CUDA 환경에서만 의미가 있으므로 CPU에서는 자동으로 꺼진다.
    use_amp = bool(cfg["train"].get("amp", False)) and device == "cuda"
    print(f"[i] device={device}  amp={use_amp}")

    task = cfg["task"]
    classes = cfg["classes"]
    img_size = cfg["data"]["img_size"]

    # train/val은 같은 클래스 인덱스 체계를 써야 하므로
    # train_loader에서 만든 c2i를 val 쪽에도 그대로 사용한다.
    train_loader, c2i = make_loader(
        cfg["data"]["train_csv"], task, classes, img_size,
        train=True, batch_size=cfg["train"]["batch_size"],
        num_workers=cfg["train"]["num_workers"],
    )
    val_loader, _ = make_loader(
        cfg["data"]["val_csv"], task, classes, img_size,
        train=False, batch_size=cfg["train"]["batch_size"],
        num_workers=cfg["train"]["num_workers"],
    )

    # backbone 이름, dropout, hidden 크기 모두 config에서 바꿀 수 있다.
    model = CoffeeClassifier(
        backbone=cfg["model"]["name"],
        n_classes=len(classes),
        pretrained=cfg["model"]["pretrained"],
        dropout=cfg["model"]["dropout"],
        hidden=cfg["model"]["hidden"],
    ).to(device)

    # class weight는 train split 기준으로 계산한다.
    cls_w = compute_class_weights(cfg["data"]["train_csv"], task, c2i).to(device)
    criterion = cross_entropy_loss(weight=cls_w,
                                   label_smoothing=cfg["train"]["label_smoothing"])

    # AdamW + CosineAnnealingLR이 현재 프로젝트의 기본 학습 조합이다.
    optimizer = AdamW(model.parameters(), lr=cfg["train"]["lr"],
                      weight_decay=cfg["train"]["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg["train"]["epochs"])
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    logger = TBLogger(cfg["paths"]["log_dir"])
    ckpt_dir = Path(cfg["paths"]["ckpt_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    warmup = cfg["train"].get("warmup_epochs", 0)
    if warmup > 0:
        # Stage 1:
        # pretrained backbone은 그대로 두고 마지막 head만 먼저 맞춘다.
        # 클래스 수가 바뀌는 전이학습에서 흔히 쓰는 안정화 단계다.
        model.freeze_backbone(True)
        print(f"[i] backbone frozen for {warmup} warm-up epoch(s)")

    # 정확도보다 macro F1을 더 중요하게 본 이유:
    # 클래스 불균형이 있는 데이터에서 소수 클래스 성능까지 함께 보기 위해서다.
    best_f1 = -1.0
    bad = 0
    for epoch in range(1, cfg["train"]["epochs"] + 1):
        if epoch == warmup + 1 and warmup > 0:
            # warm-up이 끝나면 backbone까지 함께 학습한다.
            model.freeze_backbone(False)
            print(f"[i] backbone unfrozen at epoch {epoch}")

        model.train()
        running = 0.0
        for x, y in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
            x, y = x.to(device), y.to(device)
            # set_to_none=True는 약간 더 효율적으로 gradient를 초기화한다.
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                # model(x)는 logits를 내고, criterion이 softmax 포함 CE를 계산한다.
                loss = criterion(model(x), y)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); optimizer.step()
            running += loss.item() * x.size(0)
        # scheduler는 epoch 단위로 한 번씩만 갱신한다.
        scheduler.step()

        train_loss = running / len(train_loader.dataset)
        # 검증셋 성능을 보고 checkpoint 저장 여부와 early stopping을 결정한다.
        metrics = evaluate(model, val_loader, device)
        print(f"[E{epoch:02d}] loss={train_loss:.4f}  "
              f"val_acc={metrics['acc']:.4f}  val_f1={metrics['macro_f1']:.4f}")
        logger.log({"train/loss": train_loss,
                    "val/acc": metrics["acc"],
                    "val/macro_f1": metrics["macro_f1"],
                    "lr": optimizer.param_groups[0]["lr"]}, epoch)

        if metrics["macro_f1"] > best_f1:
            # 최고 성능일 때만 저장하므로 checkpoints 폴더에는
            # "가장 좋았던 모델"이 남는다.
            best_f1 = metrics["macro_f1"]
            bad = 0
            ckpt_path = ckpt_dir / f"best_{task}.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "class_to_idx": c2i,
                "config": cfg,
                "epoch": epoch,
                "val_macro_f1": best_f1,
            }, ckpt_path)
            print(f"   ↳ saved {ckpt_path} (f1={best_f1:.4f})")
        else:
            bad += 1
            if bad >= cfg["train"]["early_stop_patience"]:
                # patience 동안 개선이 없으면 과적합 구간으로 보고 종료한다.
                print(f"[i] early stopping at epoch {epoch}")
                break

    logger.close()
    print(f"[DONE] best val_macro_f1 = {best_f1:.4f}")


## Step 7. Run Entry Point

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
if __name__ == "__main__":
    main()


## 실행 파라미터 가이드

- 이 파일은 원래 CLI 인자(argparse) 기반으로 동작합니다.
- 노트북에서는 인자 대신 아래처럼 변수 셀을 만들어 실행하세요.


In [ ]:
# 예시 파라미터 셀
CONFIG_PATH = 'configs/default.yaml'
CKPT_PATH = None
